# Mục 3.4 — Thực hành khai phá luật kết hợp

Sinh tu ban thao giao trinh. Chay tuan tu tu tren xuong; moi o phu thuoc cac o truoc no.


## 3.4. Thực hành

Mục 3.4 dành cho thực hành: vận dụng kiến thức đã học trên một tập dữ liệu giỏ hàng thực tế bằng Python với thư viện mlxtend, đi từ chuẩn bị dữ liệu đến diễn giải luật. Người học muốn làm cùng bài này trên giao diện đồ họa của WEKA đọc tệp 02-thuc-hanh-chuong-3.md trong thư mục weka của kho học phần (Mục A.3).

Khai phá luật kết hợp trong Python chỉ cần ba thành phần: pandas để tổ chức dữ liệu dạng bảng, scipy.io.arff để đọc tệp dữ liệu gốc, và mlxtend.frequent_patterns cung cấp sẵn Apriori, FP-Growth cùng hàm sinh luật. Mục này hướng dẫn từng bước trên cùng một tập dữ liệu, sau đó mở rộng sang dữ liệu có thuộc tính số.

### Tài nguyên đi kèm

Các ví dụ Python cần thiết được in trong giáo trình. Tài nguyên bổ sung có thể được giảng viên cung cấp tại https://github.com/FITHOUVN/Machine-Learning-Introduction. Kiểm tra danh sách tệp và phiên bản gói trước khi tải; không coi các thư mục notebook, mã mẫu và dữ liệu theo chương là đã có sẵn.

Tập dữ liệu supermarket.arff dùng trong bài thực hành chính được tải trực tiếp từ kho mã công khai của Đại học Waikato, không cần cài đặt WEKA. Cách tải trình bày ở Mục 3.4.1.

## 3.4.1. Chuẩn bị dữ liệu và thư viện

Mục này chuẩn bị ba thứ cho cả Mục 3.4: môi trường Python, tệp dữ liệu, và ma trận nhị phân mà mlxtend yêu cầu ở đầu vào.

### Yêu cầu môi trường

Dùng môi trường thống nhất ở Phụ lục B: Python 3.11 hoặc 3.12, pandas 2.2.3, cùng SciPy và mlxtend. Cài trong môi trường ảo và lưu lại phiên bản thực tế để tái lập kết quả.

**Kết quả đối chiếu**

```
python -m pip install "pandas==2.2.3" scipy mlxtend matplotlib
python -m pip freeze > requirements-lock.txt
```

### Bước 1. Tải tập dữ liệu Supermarket

Tệp supermarket.arff nằm trong kho mã của Đại học Waikato. Có thể tải bằng trình duyệt hoặc bằng đoạn mã sau; đoạn mã chỉ tải một lần và in kích thước tệp để đối chiếu.



In [ ]:
import urllib.request
from pathlib import Path

URL = ('https://git.cms.waikato.ac.nz/weka/weka/-/raw/main/'
       'trunk/wekadocs/data/supermarket.arff')
dich = Path('datasets/raw/supermarket.arff')
dich.parent.mkdir(parents=True, exist_ok=True)
if not dich.exists():
    urllib.request.urlretrieve(URL, dich)
print('Kich thuoc tep (byte):', dich.stat().st_size)


### Đối chiếu kết quả Bước 1

- Kích thước tệp: 2 025 871 byte. Nếu số này khác, tệp đã tải thiếu hoặc địa chỉ đã đổi.

- Địa chỉ dự phòng khi kho trên không truy cập được: https://raw.githubusercontent.com/Waikato/weka-3.8/master/wekadocs/data/supermarket.arff

### Bước 2. Đọc tệp ARFF và kiểm tra dữ liệu

ARFF là định dạng văn bản có phần khai báo thuộc tính ở đầu tệp. SciPy đọc được định dạng này; các giá trị phân loại được trả về dưới dạng chuỗi byte nên cần giải mã về chuỗi ký tự trước khi so sánh.



In [ ]:
import pandas as pd
from scipy.io import arff

d, meta = arff.loadarff('datasets/raw/supermarket.arff')
df = pd.DataFrame(d)
for c in df.columns:
    df[c] = df[c].str.decode('utf-8')

print('Kich thuoc:', df.shape)
print('So thuoc tinh:', len(meta.names()))
print('Thuoc tinh cuoi:', meta.names()[-1])
print(df.iloc[:3, :4])


### Đối chiếu kết quả Bước 2

- Kích thước tập dữ liệu: (4627, 217). Tập dữ liệu gồm 4 627 giao dịch của một chuỗi siêu thị New Zealand, với 216 loại sản phẩm và một thuộc tính total cho biết giỏ hàng thuộc nhóm giá trị thấp hay cao.

- Mỗi ô của 216 cột sản phẩm nhận giá trị t nếu sản phẩm có trong giao dịch, và ? nếu không có. Đây là quy ước của tệp gốc: giá trị thiếu được dùng để biểu thị "không mua", chứ không phải "không biết".

### Bước 3. Mã hóa thành ma trận nhị phân

Hàm apriori và fpgrowth của mlxtend nhận đầu vào là một DataFrame mà mỗi dòng là một giao dịch, mỗi cột là một mục, giá trị là True hoặc False. Vì vậy cần chuyển t thành True, còn ? thành False. Thuộc tính total không phải mục hàng hóa mà là một thuộc tính phân loại hai giá trị, nên được tách thành hai mục riêng total=high và total=low theo đúng cách xử lý luật kết hợp nhiều chiều ở Mục 3.2.3.



In [ ]:
cot_mat_hang = [c for c in df.columns if c != 'total']
X = pd.DataFrame({c: (df[c] == 't') for c in cot_mat_hang})
X['total=high'] = (df['total'] == 'high')
X['total=low'] = (df['total'] == 'low')

print('Kich thuoc ma tran nhi phan:', X.shape)
print('Kieu du lieu:', X.dtypes.unique())


### Đối chiếu kết quả Bước 3

- Kích thước ma trận nhị phân: (4627, 218), gồm 216 mục hàng hóa và 2 mục sinh ra từ thuộc tính total.

- Kiểu dữ liệu của mọi cột là bool. Nếu còn cột kiểu object hoặc float, mlxtend vẫn chạy nhưng chậm hơn nhiều và có thể phát cảnh báo; hãy ép về bool trước khi khai phá.

### Ba hàm sẽ dùng trong cả Mục 3.4

- apriori(X, ...): cài đặt Apriori của Mục 3.2.1, trả về bảng tập phổ biến gồm hai cột support và itemsets.

- fpgrowth(X, ...): cài đặt FP-Growth của Mục 3.2.2, cùng chữ ký và cùng kết quả với apriori.

- association_rules(tap_pho_bien, ...): sinh luật từ bảng tập phổ biến và tính sẵn các chỉ số Support, Confidence, Lift, Leverage, Conviction đã trình bày ở Mục 3.3.1.

Cả ba hàm đều nằm trong mlxtend.frequent_patterns, nên một lệnh nhập là đủ:



In [ ]:
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules


Bảng 3.9 tổng kết các tham số cần biết trong khuôn khổ nhập môn.

Bảng 3.9. Các tham số chính của apriori, fpgrowth và association_rules trong mlxtend

## 3.4.2. Bài thực hành: khai phá luật trên tập dữ liệu Supermarket

### Các bước thực hiện

Bài thực hành tiếp nối trực tiếp Mục 3.4.1: biến X là ma trận nhị phân (4627, 218) đã dựng ở Bước 3.

### Bước 4. Khảo sát mức phổ biến của từng mục

Trước khi chọn ngưỡng, cần biết các mục phân bố thế nào. Vì mọi cột đều là bool, trung bình của một cột chính là Support của mục đó.



In [ ]:
dem_muc = X.sum().sort_values(ascending=False)      # so giao dich, so nguyen
support_muc = dem_muc / len(X)                      # Support, ty le

print('10 muc pho bien nhat (so giao dich va Support):')
print(pd.DataFrame({'so_gd': dem_muc.head(10),
                    'support': support_muc.head(10).round(4)}).to_string())
print('So muc co tu 694 giao dich tro len:', (dem_muc >= 694).sum())


Đếm theo số giao dịch trước rồi mới chia cho N, chứ không so sánh trực tiếp hai số thực, để tránh sai số làm tròn ở đúng ranh giới ngưỡng. Lý do vì sao mốc là 694 chứ không phải 0,15 được giải thích ở Bước 5.

### Đối chiếu kết quả Bước 4

- Số mục có từ 694 giao dịch trở lên: 44. Con số này phải bằng đúng L(1) thu được ở Bước 6, vì tập phổ biến cỡ 1 chính là các mục đơn lẻ đạt ngưỡng. Nếu hai số lệch nhau thì ngưỡng dùng ở hai chỗ chưa thống nhất.

- bread and cake có 3 330 giao dịch, Support ≈ 0,7197.

Một số sản phẩm như bread and cake, vegetables, milk xuất hiện trong phần lớn giao dịch; phần lớn các sản phẩm còn lại rất hiếm. Riêng bread and cake có mặt trong 3 330 giao dịch, tức Support ≈ 0,7197, hơn hai phần ba tập dữ liệu. Hãy ghi lại con số này: nó chính là tần suất nền dùng để diễn giải Lift ở Bước 7. Phân bố lệch như vậy quyết định ngưỡng hợp lý và cũng báo trước một hiện tượng sẽ gặp ở Bước 7: các mục phổ biến nhất dễ chiếm vế phải của hầu hết luật.

### Bước 5. Chọn ngưỡng Support và lưu ý về cách tính ngưỡng

Bài thực hành dùng ngưỡng Support ứng với 694 giao dịch và ngưỡng Confidence 0,9. Mức này cao hơn khoảng 0,01 đến 0,05 đã khuyến nghị ở Mục 3.2.1 cho dữ liệu hàng nghìn giao dịch; ở đây giữ ngưỡng cao để lần chạy đầu tiên nhanh và dễ đối chiếu, việc hạ dần ngưỡng được để lại cho phần bài tập.



In [ ]:
N = len(X)                       # 4627 giao dich
nguong = 694 / N                 # 0.14998919...
print(f'Nguong Support: {nguong:.8f}  tuong ung {694} giao dich')


### Ghi chú về ngưỡng tính theo tỷ lệ và ngưỡng tính theo số đếm

Đây là một khác biệt nhỏ nhưng có hậu quả thật, cần nắm trước khi so sánh kết quả với bất kỳ tài liệu nào. Với 4 627 giao dịch, ngưỡng 0,15 tương ứng 0,15 × 4627 = 694,05 giao dịch. Một số công cụ làm tròn xuống thành 694 rồi lọc theo số đếm, nghĩa là giữ lại cả các tập có đúng 694 giao dịch hỗ trợ. Hàm apriori của mlxtend lọc theo tỷ lệ, nên khi đặt min_support=0.15 thì điều kiện trở thành số giao dịch lớn hơn hoặc bằng 694,05, tức là từ 695 trở lên. Chênh lệch một giao dịch này làm mất 7 tập phổ biến: 1 tập ở cỡ 2, 1 tập ở cỡ 3, 4 tập ở cỡ 4 và 1 tập ở cỡ 5. Vì vậy trong bài này ngưỡng được viết là 694 / N chứ không phải 0,15. Nguyên tắc chung: khi cần một ngưỡng theo số giao dịch, hãy viết thẳng phân số so_giao_dich / N thay vì gõ một số thập phân đã làm tròn.

### Bước 6. Khai phá tập phổ biến bằng Apriori



In [ ]:
import time
from mlxtend.frequent_patterns import apriori

t0 = time.perf_counter()
tap_pho_bien = apriori(X, min_support=nguong,
                       use_colnames=True, max_len=6)
t_apriori = time.perf_counter() - t0

co = tap_pho_bien['itemsets'].apply(len)
print('So tap pho bien theo co:')
print(co.value_counts().sort_index().to_string())
print('Tong so tap pho bien:', len(tap_pho_bien))
print(f'Thoi gian chay Apriori: {t_apriori:.2f} giay')


Tham số max_len=6 giới hạn cỡ tập mục được xét, giúp lần chạy đầu nhanh hơn. Ở ngưỡng này, giới hạn đó không làm mất kết quả nào: cỡ 6 chỉ có duy nhất một tập phổ biến, mà muốn sinh một ứng viên cỡ 7 thì cần ít nhất hai tập phổ biến cỡ 6 dùng chung 5 mục, nên quá trình tìm kiếm dừng ngay sau cỡ 6.

### Đối chiếu kết quả Bước 6

**Kết quả đối chiếu**

```
So tap pho bien theo co:
1     44
2    380
3    910
4    633
5    105
6      1
Tong so tap pho bien: 2073
```

Phân bố này, vẽ ở Hình 3.7, là hình ảnh trực quan của tính chất Apriori đã học ở Mục 3.2.1: số tập phổ biến tăng theo cỡ cho đến một đỉnh, sau đó giảm nhanh vì điều kiện Support ngày càng khó thỏa. Ở đây đỉnh nằm ở cỡ 3 với 910 tập.



In [ ]:
import matplotlib.pyplot as plt

dem = co.value_counts().sort_index()
plt.figure(figsize=(7, 4))
plt.bar(dem.index.astype(str), dem.values)
for i, y in enumerate(dem.values):
    plt.text(i, y, str(y), ha='center', va='bottom')
plt.xlabel('Co cua tap muc k')
plt.ylabel('So tap pho bien L(k)')
plt.title('So tap pho bien theo co tren Supermarket')
plt.tight_layout(); plt.show()


Hình 3.7. Số tập phổ biến theo cỡ trên tập dữ liệu Supermarket (ngưỡng Support 694/4627)

### Bước 7. Sinh luật kết hợp và đọc kết quả

Hàm association_rules nhận bảng tập phổ biến vừa thu được, lọc theo chỉ số chọn ở metric và trả về một DataFrame. Vì DataFrame chỉ ghi Support dưới dạng tỷ lệ, ta nhân lại với N để có số giao dịch, cho dễ đối chiếu và dễ diễn giải.



In [ ]:
from mlxtend.frequent_patterns import association_rules

luat = association_rules(tap_pho_bien, metric='confidence',
                         min_threshold=0.9)
print('So luat dat nguong Confidence 0.9:', len(luat))

luat['gd_tien_de'] = (luat['antecedent support'] * N).round().astype(int)
luat['gd_ca_luat'] = (luat['support'] * N).round().astype(int)

cot = ['antecedents', 'consequents', 'gd_tien_de', 'gd_ca_luat',
       'support', 'confidence', 'lift', 'leverage', 'conviction']
top = luat.sort_values('confidence', ascending=False).head(10)
print(top[cot].to_string(index=False))


### Đối chiếu kết quả Bước 7

Hai luật mạnh nhất, trình bày lại cho gọn:

**Kết quả đối chiếu**

```
1. biscuits, frozen foods, fruit, total=high   788  ==>  bread and cake   723
   Support 0.1563   Confidence 0.9175   Lift 1.2749   Leverage 0.0337   Conviction 3.3982

2. baking needs, biscuits, fruit, total=high   760  ==>  bread and cake   696
   Support 0.1504   Confidence 0.9158   Lift 1.2725   Leverage 0.0322   Conviction 3.3287
```

### Diễn giải

Trong 788 giao dịch thỏa vế trái của luật 1, có 723 giao dịch chứa bread and cake: Confidence = 723/788 ≈ 0,9175, làm tròn thành 0,92. Support của cả luật là 723/4627 ≈ 0,1563.

Mặt hàng bread and cake xuất hiện trong 3 330 giao dịch, tức Support nền là 3330/4627 ≈ 0,7197. Lift = 0,9175/0,7197 ≈ 1,27 cho biết tần suất của vế phải trong nhóm thỏa vế trái cao hơn khoảng 27% so với tần suất nền. Chỉ số này cho thấy liên hệ thuận trong dữ liệu, không tự chứng minh ý nghĩa thống kê hoặc quan hệ nhân quả. Leverage 0,0337 là phần Support dôi ra so với trường hợp hai vế độc lập: 0,1563 − 0,1703 × 0,7197 ≈ 0,0337, một lượng nhỏ nhưng dương. Conviction 3,40 nói rằng nếu vế trái và vế phải độc lập thì số trường hợp luật bị vi phạm sẽ lớn hơn khoảng 3,4 lần so với thực tế quan sát được.

### Lưu ý: cùng một chỉ số có thể được công cụ khác tính theo quy ước khác

Tập dữ liệu Supermarket vốn đi kèm công cụ WEKA và kết quả chạy bằng WEKA được in trong nhiều tài liệu, nên người học dễ có dịp đối chiếu. Khi đối chiếu, bốn chỉ số Support, Confidence, Lift và Leverage trùng khít tới từng chữ số; riêng Conviction thì không: WEKA in 3,35 và 3,28 cho hai luật trên, còn mlxtend cho 3,40 và 3,33. Đây là chỗ duy nhất trong cả Mục 3.4 mà con số khác với kết quả WEKA.

Nguyên nhân không phải là lỗi của bên nào: hai công cụ dùng hai quy ước tính Conviction khác nhau. mlxtend tính đúng theo định nghĩa đã học ở Mục 3.3.1, tức Conviction = (1 − Support(Y)) / (1 − Confidence) = (1 − 0,7197) / (1 − 0,9175) = 0,280312/0,082487 ≈ 3,3982. Bài học rút ra áp dụng cho mọi chỉ số, không riêng Conviction: khi một con số lệch so với tài liệu tham khảo, trước khi kết luận mình làm sai, hãy kiểm lại công thức mà mỗi công cụ thực sự dùng. Vì vậy báo cáo kết quả phải ghi rõ công cụ và phiên bản đã dùng; chỉ nói "Conviction bằng 3,4" là chưa đủ để người khác tái lập.

### Kết quả và nhận xét từ thực hành

Nếu nhiều luật có vế phải là mặt hàng phổ biến, hãy đổi chỉ số lọc sang Lift bằng metric='lift', hoặc giữ nguyên cách lọc theo Confidence rồi sắp xếp và sàng lại bảng luat theo Lift cùng số giao dịch hỗ trợ:



In [ ]:
loc = luat[(luat['lift'] >= 1.3) & (luat['gd_ca_luat'] >= 800)]
print(loc.sort_values('lift', ascending=False)[cot].head(10).to_string(index=False))


Việc lọc hậu kỳ trên DataFrame là một thuận lợi của cách làm bằng Python: mọi chỉ số đã được tính sẵn thành cột, nên thay tiêu chí sàng lọc không cần chạy lại thuật toán. Ngược lại, muốn đổi ngưỡng Support thì buộc phải chạy lại apriori, vì ngưỡng đó quyết định tập phổ biến.

Các luật chứa total=high mô tả liên hệ với giỏ hàng giá trị cao; cần kiểm tra tính khả dụng của thuộc tính này trước khi dùng để đề xuất khuyến mãi, vì giá trị giỏ hàng chỉ biết được sau khi khách đã thanh toán.

## 3.4.3. So sánh Apriori và FP-Growth bằng mlxtend

Sau Apriori, nên lặp lại bài thực hành với FP-Growth. Trong mlxtend, hai thuật toán có cùng chữ ký hàm, nên chỉ cần đổi tên hàm; đây là cách so sánh công bằng nhất vì mọi tham số còn lại giữ nguyên.



In [ ]:
import time
from mlxtend.frequent_patterns import apriori, fpgrowth

t0 = time.perf_counter()
tap_ap = apriori(X, min_support=nguong, use_colnames=True, max_len=6)
t_ap = time.perf_counter() - t0

t0 = time.perf_counter()
tap_fp = fpgrowth(X, min_support=nguong, use_colnames=True, max_len=6)
t_fp = time.perf_counter() - t0

print('So tap pho bien Apriori  :', len(tap_ap))
print('So tap pho bien FP-Growth:', len(tap_fp))
print('Hai ket qua trung nhau   :',
      set(tap_ap['itemsets']) == set(tap_fp['itemsets']))
print(f'Thoi gian Apriori  : {t_ap:.2f} giay')
print(f'Thoi gian FP-Growth: {t_fp:.2f} giay')


### So sánh kết quả

Hai thuật toán cho cùng 2 073 tập phổ biến và cùng một tập hợp các tập mục, đúng như lý thuyết ở Mục 3.2.2: FP-Growth là một cách tổ chức lại quá trình tìm kiếm, không phải một định nghĩa khác về tập phổ biến. Điều cần chú ý khi lập trình là thứ tự các dòng trong hai DataFrame khác nhau, vì hai thuật toán sinh tập mục theo trật tự khác nhau. Do đó phải so sánh dưới dạng tập hợp các frozenset như trong đoạn mã trên; so sánh trực tiếp hai DataFrame bằng equals sẽ cho kết quả sai lệch.

Khác biệt thực sự nằm ở thời gian chạy. Hãy chạy đoạn mã trên và ghi lại hai con số t_ap và t_fp trên máy của mình, kèm cấu hình máy và phiên bản mlxtend. Lưu ý rằng một lần đo đơn lẻ chịu ảnh hưởng của tải máy; nên chạy lại ba lần và lấy giá trị nhỏ nhất. Chênh lệch giữa hai thuật toán phụ thuộc cả vào ngưỡng Support: ngưỡng càng thấp, số ứng viên mà Apriori phải sinh càng lớn, nên lợi thế dần nghiêng về FP-Growth. Cần lưu ý rằng ở chính ngưỡng của bài thực hành này, chiều của chênh lệch ngược với trông đợi thông thường. Đây chính là nội dung bài tập mở rộng ở cuối Mục 3.4.

### Số đo thực tế trên tập dữ liệu Supermarket

Các số dưới đây đo trên cùng một máy với mlxtend 0.25.0, tham số max_len=6 như trong đoạn mã, mỗi mức chạy ba lần và lấy giá trị nhỏ nhất. Số tuyệt đối sẽ khác trên máy khác, nhưng tương quan giữa hai thuật toán thì không.

- Ngưỡng 694/4627, tức Support 0,1500, cho 2 073 tập phổ biến: Apriori 0,09 giây, FP-Growth 0,59 giây. Apriori nhanh hơn khoảng 6,5 lần.

- Ngưỡng 500/4627, tức Support 0,1081, cho 7 655 tập phổ biến: Apriori 0,37 giây, FP-Growth 1,35 giây. Apriori nhanh hơn khoảng 3,7 lần.

- Ngưỡng 400/4627, tức Support 0,0864, cho 17 975 tập phổ biến: Apriori 0,73 giây, FP-Growth 2,13 giây. Apriori nhanh hơn khoảng 2,9 lần.

- Ngưỡng 300/4627, tức Support 0,0648, cho 50 658 tập phổ biến: Apriori 2,86 giây, FP-Growth 3,84 giây. Apriori nhanh hơn khoảng 1,3 lần.

- Ngưỡng 231/4627, tức Support 0,0499, cho 118 301 tập phổ biến: Apriori 8,85 giây, FP-Growth 8,25 giây. FP-Growth bắt đầu nhanh hơn.

Chạy đoạn mã ở ngưỡng của bài thực hành, sinh viên sẽ thấy Apriori nhanh hơn FP-Growth. Đây không phải lỗi cài đặt cũng không phải sai số đo, mà đúng bằng điều Mục 3.2.2 đã cảnh báo ở Trường hợp sai 2: tập dữ liệu chỉ có 4 627 giao dịch và ngưỡng Support lại cao, nên FP-Tree gần như không nén được gì, và chi phí dựng cây rồi khai thác đệ quy vượt quá chi phí quét dữ liệu của Apriori. Kết quả này cũng khớp với quy tắc chọn thuật toán ở cùng mục: dưới vài nghìn giao dịch thì Apriori vẫn là lựa chọn hợp lý.

Xu hướng mà lý thuyết nêu vẫn đúng: ngưỡng càng thấp thì lợi thế càng nghiêng về FP-Growth, và trên tập dữ liệu này điểm giao nhau nằm quanh Support 0,05. Ưu thế thực sự của FP-Growth chỉ bộc lộ trên dữ liệu lớn hơn nhiều, đúng như Trường hợp chọn đúng đã mô tả. Vì vậy, kết luận cần rút ra từ bài thực hành không phải là thuật toán nào nhanh hơn, mà là tốc độ phụ thuộc vào cỡ dữ liệu và ngưỡng Support, nên phải đo trên chính dữ liệu của mình trước khi chọn.

Vì tập phổ biến giống nhau, luật sinh ra từ hai bảng cũng giống nhau. Có thể kiểm chứng bằng cách so sánh hai tập luật sau khi chuẩn hóa về khóa so sánh được:



In [ ]:
def khoa_luat(bang):
    return {(frozenset(a), frozenset(c))
            for a, c in zip(bang['antecedents'], bang['consequents'])}

luat_ap = association_rules(tap_ap, metric='confidence', min_threshold=0.9)
luat_fp = association_rules(tap_fp, metric='confidence', min_threshold=0.9)
print('Hai tap luat trung nhau:', khoa_luat(luat_ap) == khoa_luat(luat_fp))


## 3.4.4. Xử lý dữ liệu số bằng rời rạc hóa

Mục 3.2.3 đã chỉ ra rằng các thuộc tính số cần được rời rạc hóa trước khi đưa vào khai phá luật kết hợp, vì luật kết hợp chỉ làm việc với các mục có hoặc không. Trong Python, công việc này gồm hai thao tác: chia giá trị số thành các khoảng bằng pd.cut hoặc pd.qcut, rồi mã hóa mỗi khoảng thành một cột nhị phân bằng pd.get_dummies.

### Hai cách chia khoảng

- pd.cut: chia theo độ rộng. Các khoảng có độ rộng bằng nhau, số bản ghi mỗi khoảng khác nhau. Dùng khi ranh giới có nghĩa sẵn trong lĩnh vực, ví dụ các mốc tuổi 18, 22, 30.

- pd.qcut: chia theo phân vị. Mỗi khoảng chứa xấp xỉ cùng số bản ghi, độ rộng khác nhau. Dùng khi phân bố lệch và muốn mỗi khoảng đủ bản ghi để Support không quá nhỏ.

### Bài thực hành rời rạc hóa

Dùng lại tập dữ liệu sinh viên của Mục 2.7, tệp datasets/raw/dmml_sinhvien.csv, gồm 200 bản ghi với các cột số Tuoi, DiemTB_HK1, SoTinChi, SoBuoiVangMat và các cột phân loại NganhHoc, HinhThucDangKy, KetQua.



In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

bang = pd.read_csv('datasets/raw/dmml_sinhvien.csv')

# Buoc 1: xu ly thieu va loai ba gia tri nhap sai da biet tu Muc 2.7
bang['Tuoi'] = bang['Tuoi'].fillna(bang['Tuoi'].median())
bang = bang[(bang['Tuoi'] >= 15) & (bang['Tuoi'] <= 30)].copy()
print('So ban ghi dung cho khai pha luat:', len(bang))

# Buoc 2: roi rac hoa
# pd.cut - moc do tuoi co y nghia san
bang['Tuoi_nhom'] = pd.cut(bang['Tuoi'], bins=[14, 19, 21, 30],
                           labels=['tuoi<=19', 'tuoi 20-21', 'tuoi>=22'])
# pd.qcut - chia ba nhom co so ban ghi xap xi bang nhau
bang['Diem_nhom'] = pd.qcut(bang['DiemTB_HK1'], q=3,
                            labels=['diem thap', 'diem trung binh',
                                    'diem cao'])
bang['Vang_nhom'] = pd.qcut(bang['SoBuoiVangMat'], q=2,
                            labels=['vang it', 'vang nhieu'],
                            duplicates='drop')
print(bang['Tuoi_nhom'].value_counts().sort_index().to_string())
print(bang['Diem_nhom'].value_counts().sort_index().to_string())

# Buoc 3: ma hoa nhi phan. Moi khoang va moi gia tri phan loai
# tro thanh mot muc
cot_muc = ['Tuoi_nhom', 'Diem_nhom', 'Vang_nhom',
           'NganhHoc', 'HinhThucDangKy', 'KetQua']
G = pd.get_dummies(bang[cot_muc], prefix_sep='=').astype(bool)
print('Kich thuoc ma tran muc:', G.shape)
print('Kiem tra khong mat ban ghi:', G.sum(axis=1).unique())

# Buoc 4: khai pha nhu binh thuong
tap = apriori(G, min_support=0.10, use_colnames=True)
luat_dl = association_rules(tap, metric='confidence', min_threshold=0.7)
luat_dl = luat_dl.sort_values('lift', ascending=False)
print(luat_dl[['antecedents', 'consequents',
               'support', 'confidence', 'lift']].head(10).to_string(index=False))


### Đối chiếu kết quả

- Số bản ghi dùng cho khai phá luật: 197, trùng với kết quả Phần A của Mục 2.7 sau khi điền giá trị thiếu và loại ba giá trị nhập sai 5, 99 và 150.

- G.sum(axis=1).unique() phải trả về đúng một giá trị là 6, nghĩa là mỗi bản ghi đóng góp đúng một mục cho mỗi trong sáu cột nguồn. Nếu xuất hiện giá trị nhỏ hơn 6, có bản ghi đã rơi ra ngoài các khoảng của pd.cut và bị mất mục một cách âm thầm.

### Đọc luật định lượng

Mỗi luật thu được là một luật định lượng theo nghĩa của Mục 3.2.3, chỉ khác ở cách viết tên mục: Diem_nhom=diem cao thay cho điểm ∈ [3,3; 4,0]. Muốn in ra ranh giới thật của từng khoảng, hãy giữ lại đối tượng khoảng thay vì nhãn chữ:



In [ ]:
_, moc = pd.qcut(bang['DiemTB_HK1'], q=3, retbins=True)
print('Moc chia ba nhom diem:', moc.round(3))


### Những điểm cần cẩn trọng

- Số khoảng càng nhiều thì Support của mỗi mục càng nhỏ, nên ngưỡng Support phải hạ theo. Với ba khoảng, Support tối đa của một mục khoảng chỉ xấp xỉ 1/3; đặt min_support=0.4 sẽ không thu được luật nào.

- pd.cut trả về giá trị thiếu cho những bản ghi nằm ngoài các khoảng đã khai báo, và pd.get_dummies mặc định bỏ qua giá trị thiếu. Hậu quả là bản ghi đó mất mục mà không có thông báo lỗi. Luôn kiểm tra bằng phép đếm như ở Bước 3.

- pd.qcut báo lỗi khi các mốc phân vị trùng nhau, thường gặp ở cột số có ít giá trị khác biệt hoặc tập trung vào một giá trị. Tham số duplicates='drop' xử lý được tình huống này, nhưng khi đó số khoảng thực tế ít hơn q, nên không được truyền danh sách labels có số phần tử cố định.

- Cột nhãn KetQua được đưa vào ma trận mục nên luật có thể có vế phải là KetQua=Khong dat. Đây không phải phân lớp: luật kết hợp không tối ưu hóa khả năng dự báo và không được đánh giá trên tập kiểm tra độc lập. Ngoài ra nhãn này mất cân bằng khoảng 85/15, nên ngưỡng Support 0,10 đã gần sát tần suất nền của nhóm thiểu số; muốn khảo sát nhóm này phải hạ ngưỡng và chấp nhận số luật tăng nhanh.

## 3.4.5. Lưu ý chung khi thực hành

### Lỗi thường gặp

- Lỗi 1: truyền vào apriori một DataFrame chưa mã hóa nhị phân, còn kiểu object hoặc kiểu số. Hậu quả là chạy rất chậm, phát cảnh báo về hiệu năng, hoặc cho kết quả vô nghĩa vì mọi giá trị khác 0 đều được coi là có mặt. Cách khắc phục: mã hóa bằng TransactionEncoder (Mục 3.2.1) hoặc pd.get_dummies, rồi ép astype(bool). Nếu bảng có thuộc tính số thì phải rời rạc hóa trước theo Mục 3.4.4; không có thuộc tính số nào tự trở thành mục được.

- Lỗi 2: đặt min_support quá thấp khiến chương trình chạy rất lâu hoặc hết bộ nhớ. Cách khắc phục: bắt đầu từ ngưỡng cao rồi giảm dần; đặt max_len để giới hạn cỡ tập mục; dùng fpgrowth thay cho apriori; nếu vẫn không được, thử trên một mẫu nhỏ của tập dữ liệu trước.

- Lỗi 3: nhầm giữa ngưỡng tính theo tỷ lệ và ngưỡng tính theo số giao dịch. min_support của mlxtend luôn là tỷ lệ; muốn ngưỡng ứng với một số giao dịch cụ thể thì viết so_giao_dich / N như đã giải thích ở Mục 3.4.2. Đây là nguyên nhân phổ biến nhất của việc kết quả lệch một vài tập so với tài liệu tham khảo.

- Lỗi 4: đặt min_threshold quá cao, ví dụ 0,95 cho Confidence, khiến association_rules trả về bảng rỗng và các lệnh in phía sau báo lỗi thiếu cột. Cách khắc phục: kiểm tra len(luat) trước khi xử lý tiếp, rồi hạ ngưỡng dần.

- Lỗi 5: dùng use_colnames=False, tức giá trị mặc định, khiến tập mục hiện ra dưới dạng chỉ số cột và không thể đọc. Cách khắc phục: luôn đặt use_colnames=True.

- Lỗi 6: luật chủ yếu có vế phải là mặt hàng phổ biến. Cách khắc phục: đổi metric sang lift, hoặc sàng lại bảng luật theo Lift và Leverage cùng số giao dịch hỗ trợ. Không hạ ngưỡng Support với mục đích loại mặt hàng phổ biến, vì hạ ngưỡng chỉ làm số luật tăng thêm.

- Lỗi 7: luật có Confidence đúng bằng 1 làm ô tương ứng ở cột conviction nhận giá trị inf, khiến các phép sắp xếp hay thống kê trên cột này cho kết quả lạ. Đây là hệ quả trực tiếp của định nghĩa: mẫu số 1 − Confidence bằng 0. Cách khắc phục: xử lý riêng các luật này, và nhớ rằng Confidence bằng 1 trên một tập dữ liệu hữu hạn thường là dấu hiệu của luật tầm thường hoặc luật dư thừa đã bàn ở Mục 3.3.

- Lỗi 8: chữ ký hàm association_rules có thay đổi giữa một số phiên bản mlxtend. Nếu gặp lỗi thiếu đối số bắt buộc, hãy tra help(association_rules) của đúng phiên bản đang cài thay vì sao chép mã từ nguồn khác. Ghi lại phiên bản thực tế trong requirements-lock.txt để bài làm tái lập được.

### Hướng dẫn thực hành

Khảo sát Support của từng mục bằng X.mean() trước khi chạy, để ước lượng ngưỡng hợp lý thay vì dò ngẫu nhiên.

Bắt đầu với ngưỡng Support cao rồi giảm dần, kết hợp max_len, để tránh chờ lâu trong lúc dò ngưỡng.

Lưu kết quả để so sánh giữa các lần chạy. Hai cột antecedents và consequents chứa frozenset nên không ghi ra tệp CSV một cách sạch sẽ; cần chuyển về chuỗi trước khi lưu:



In [ ]:
from pathlib import Path
Path('ketqua').mkdir(exist_ok=True)

luat_luu = luat.copy()
for c in ['antecedents', 'consequents']:
    luat_luu[c] = luat_luu[c].apply(lambda s: ', '.join(sorted(s)))
luat_luu.to_csv('ketqua/luat_supermarket.csv', index=False)


Ghi lại ngưỡng, max_len, phiên bản gói và mã commit của gói dữ liệu cùng với kết quả. Một bảng luật không kèm ngưỡng đã dùng thì không có giá trị đối chiếu.

Luôn diễn giải kết quả với kiến thức lĩnh vực: luật có chỉ số cao chưa chắc có ý nghĩa thực tế, và ngược lại.

### Tự luyện thực hành

Yêu cầu: Chọn một trong các tập dữ liệu sau đây và thực hiện một bài khai phá luật kết hợp đầy đủ:

- (1) Online Retail. Tập dữ liệu từ UCI ML Repository: dữ liệu giao dịch online của một cửa hàng bán lẻ Anh quốc trong khoảng một năm, có khoảng nửa triệu dòng hóa đơn.

- (2) Groceries. Tập dữ liệu từ Kaggle: dữ liệu giỏ hàng tạp hóa, đơn giản hơn để bắt đầu.

- (3) Tự tạo một tập dữ liệu nhỏ (15-20 giao dịch) phản ánh một bối cảnh quen thuộc với bản thân.

Thực hiện các bước:

- (a) Mô tả tập dữ liệu đã chọn: nguồn gốc, số giao dịch, số mục, mẫu giao dịch điển hình.

- (b) Chuyển dữ liệu về ma trận nhị phân: dùng TransactionEncoder nếu dữ liệu ở dạng danh sách các giao dịch, hoặc pivot_table cùng pd.get_dummies nếu dữ liệu ở dạng bảng hóa đơn một dòng một mặt hàng. Rời rạc hóa trước nếu có thuộc tính số.

- (c) Khảo sát Support của từng mục, chọn ngưỡng Support và ngưỡng Confidence có lập luận. Ghi rõ ngưỡng theo tỷ lệ tương ứng với bao nhiêu giao dịch.

- (d) Chạy apriori, ghi nhận số tập phổ biến theo cỡ và 10 luật mạnh nhất theo Confidence.

- (e) Chạy fpgrowth với cùng min_support và cùng max_len. Kiểm tra hai tập phổ biến có trùng nhau không, đo và so sánh thời gian chạy. Lặp lại ở hai mức ngưỡng khác nhau và nhận xét ngưỡng ảnh hưởng thế nào tới chênh lệch thời gian.

- (f) Diễn giải các luật quan trọng nhất theo ngôn ngữ của lĩnh vực (kinh doanh, y tế, giáo dục, v.v.). Đề xuất ít nhất 3 ứng dụng thực tế của các luật này, và nêu rõ một giới hạn khiến luật chưa thể dùng trực tiếp để ra quyết định.

